In [8]:
from pathlib import Path

import geopandas as gpd


RAIZ_PROYECTO = Path.cwd()
RUTA_BASE = RAIZ_PROYECTO / "base" / "localidades_objetivo_sus.gpkg"
RUTA_SALIDA = RAIZ_PROYECTO / "public" / "mapa_base.geojson"

base = gpd.read_file(RUTA_BASE)

columnas_requeridas = {"cve_loc", "nom_loc", "geometry"}
columnas_faltantes = columnas_requeridas.difference(base.columns)
if columnas_faltantes:
    raise ValueError(
        f"Faltan columnas requeridas en la base: {sorted(columnas_faltantes)}"
    )

base = base[["cve_loc", "nom_loc", "geometry"]].copy()
base["cve_loc"] = base["cve_loc"].astype("string").str.strip()
base["nom_loc"] = base["nom_loc"].astype("string").str.strip()
base = base.dropna(subset=["cve_loc", "nom_loc", "geometry"])
base = base[base.geometry.is_valid & ~base.geometry.is_empty]
base = base.drop_duplicates(subset=["cve_loc"], keep="first")

if base.crs is None:
    raise ValueError("La base no tiene CRS definido; no es seguro transformar geometry.")

base = base.to_crs("EPSG:6372")
base["geometry"] = base.geometry.simplify(100, preserve_topology=True)
base = base.to_crs("EPSG:4326")
RUTA_SALIDA.parent.mkdir(parents=True, exist_ok=True)
base.to_file(RUTA_SALIDA, driver="GeoJSON")

print(f"Registros exportados: {len(base):,}")
print(f"CRS de salida: {base.crs}")
print(f"Archivo generado: {RUTA_SALIDA}")

Registros exportados: 2,390
CRS de salida: EPSG:4326
Archivo generado: c:\Users\jose.valdez\Downloads\nuevo-map\mapa-AGEB-\public\mapa_base.geojson
